# Rotulagem por similaridade

### Extração de clusters

In [2]:
import pandas as pd
import random
import numpy as np
import hdbscan
from umap import UMAP
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
from hyperopt import hp
from hyperopt import fmin, tpe, STATUS_OK, space_eval, Trials
from functools import partial

/home/andre/Documentos/Workspace/mestrado/dissertacao/venv/lib/python3.10/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2022-12-23 12:29:21.924609: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2022-12-23 12:29:22.002862: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2022-12-23 12:29:22.430044: W tensorflow/compiler/x

In [3]:
docs = pd.read_csv('../data/processed/mean_tweets.csv')
docs

,text
0,i think it s hilarious obama is trying to do s...
1,go vote for and kick it big time to save chris...
2,we picked a he to us time to turn on no
3,is about to be the sexiest in history all due ...
4,he is a to and obsessed with not
...,...
94741,todavia es posible evitar que trump llegue a l...
94742,the only thing and have in common is their hai...
94743,esta columna de enrique santos molano presenta...
94744,donald trump elected president despite usa vot...


In [4]:
from sentence_transformers import SentenceTransformer

#mpnet_model = SentenceTransformer("all-mpnet-base-v2", device="cuda")
minilm_model = SentenceTransformer("all-MiniLM-L6-v2", device="cuda")
#bert_base_model = SentenceTransformer("xlm-r-bert-base-nli-stsb-mean-tokens", device="cuda")

#mpnet_embeddings = mpnet_model.encode(docs.text, show_progress_bar=True)
minilm_embeddings = minilm_model.encode(docs.text, show_progress_bar=True)
#bert_base_embeddings = bert_base_model.encode(docs.text, show_progress_bar=True)

Batches: 100%|██████████| 2961/2961 [00:16<00:00, 179.08it/s]


In [5]:
def generate_clusters(embeddings,
                      n_neighbors,
                      n_components, 
                      min_cluster_size,
                      random_state = None):
    """
    Generate HDBSCAN cluster object after reducing embedding dimensionality with UMAP
    """
    
    umap_embeddings = (UMAP(n_neighbors=n_neighbors, 
                                n_components=n_components, 
                                metric='cosine', 
                                random_state=random_state)
                                .fit_transform(embeddings))

    clusters = hdbscan.HDBSCAN(min_cluster_size = min_cluster_size,
                               metric='euclidean', 
                               cluster_selection_method='eom').fit(umap_embeddings)

                               

    return clusters

In [6]:
import numpy as np

def score_clusters(clusters, prob_threshold = 0.05):
    """
    Returns the label count and cost of a given cluster supplied from running hdbscan
    """
    
    cluster_labels = clusters.labels_
    label_count = len(np.unique(cluster_labels))
    total_num = len(clusters.labels_)
    cost = (np.count_nonzero(clusters.probabilities_ < prob_threshold)/total_num)
    
    return label_count, cost

In [7]:
def objective(params, embeddings, clusters_lower, clusters_upper):
    """
    Objective function for hyperopt to minimize, which incorporates constraints
    on the number of clusters we want to identify
    """
    
    clusters = generate_clusters(embeddings, 
                                 n_neighbors = params['n_neighbors'], 
                                 n_components = params['n_components'], 
                                 min_cluster_size = params['min_cluster_size'],
                                 random_state = params['random_state'])
    
    label_count, cost = score_clusters(clusters, prob_threshold = 0.05)
    
    #15% penalty on the cost function if outside the desired range of groups
    if (label_count < clusters_lower) | (label_count > clusters_upper):
        penalty = 0.15 
    else:
        penalty = 0
    
    loss = cost + penalty
    
    return {'loss': loss, 'label_count': label_count, 'status': STATUS_OK}

In [8]:
def bayesian_search(embeddings, space, clusters_lower, clusters_upper, max_evals):
    """
    Perform bayseian search on hyperopt hyperparameter space to minimize objective function
    """
    
    trials = Trials()
    fmin_objective = partial(objective, embeddings=embeddings, clusters_lower=clusters_lower, clusters_upper=clusters_upper)
    best = fmin(fmin_objective,  
                space = space, 
                algo=tpe.suggest,
                max_evals=max_evals, 
                trials=trials)

    best_params = space_eval(space, best)
    print ('best:')
    print (best_params)
    print (f"label count: {trials.best_trial['result']['label_count']}")
    
    best_clusters = generate_clusters(embeddings, 
                                      n_neighbors = best_params['n_neighbors'], 
                                      n_components = best_params['n_components'], 
                                      min_cluster_size = best_params['min_cluster_size'],
                                      random_state = best_params['random_state'])
    
    return best_params, best_clusters, trials

In [9]:
hspace = {
    'n_neighbors': hp.choice('n_neighbors',range(5,20)),
    'n_components': hp.choice('n_components',range(3,20)),
    'min_cluster_size': hp.choice('min_cluster_size',range(10,80)),
    'random_state':42
}

clusters_lower=25
clusters_upper=50
max_evals = 20

In [10]:
minilm_best_params, minilm_best_clusters, minilm_trials = bayesian_search(minilm_embeddings,
                                                                       space=hspace,
                                                                       clusters_lower=clusters_lower,
                                                                       clusters_upper=clusters_upper,
                                                                       max_evals=max_evals)

 85%|████████▌ | 17/20 [22:42<03:34, 71.59s/trial, best loss: 0.5601597956641969]

/home/andre/Documentos/Workspace/mestrado/dissertacao/venv/lib/python3.10/site-packages/umap/spectral.py:260: UserWarning: WARNING: spectral initialisation failed! The eigenvector solver
failed. This is likely due to too small an eigengap. Consider
adding some noise or jitter to your data.

Falling back to random initialisation!
  warn(



100%|██████████| 20/20 [26:51<00:00, 80.58s/trial, best loss: 0.5601597956641969]
best:
{'min_cluster_size': 59, 'n_components': 3, 'n_neighbors': 9, 'random_state': 42}
label count: 176


In [ ]:
minilm_best_params